# Microsoft Global ML Building Footprints — 49-site feature table

## What this data source is

[Microsoft Global ML Building Footprints](https://github.com/microsoft/GlobalMLBuildingFootprints) is an open global dataset of building-footprint polygons detected from aerial and satellite imagery by machine-learning models. A polygon represents the estimated ground outline of one detected building. The source is therefore useful for describing the **amount and concentration of built form around a survey site**; it is not an image collection and it is not a register of cadastral parcels or confirmed building uses.

### API and access method

This data source does **not** provide a REST, ArcGIS or coordinate-based query API. The official access method is file-based: the notebook sends standard HTTPS `GET` requests to Microsoft's public Azure Blob Storage, first to download the release's `dataset-links.csv` index and then to download the indexed `.csv.gz` partition files needed for the study sites. In other words, `requests.get()` is used as an HTTP file-download client; there is no API key, authentication token or API query endpoint.

Microsoft distributes the records as line-delimited GeoJSON files, divided by country and level-9 Bing quadkey. This notebook pins the **24 July 2026** release index, whose source documentation describes footprints derived from imagery acquired between 2014 and 2024. The data are licensed under [CDLA Permissive 2.0](https://cdla.dev/permissive-2-0/). The notebook downloads only the Australia partitions required by the 49 sites and caches the unchanged source files under `data/external_raw/`.

## What information can be obtained

The source directly supplies building polygon geometry. From that geometry, this notebook can calculate how many detected footprints intersect a defined area, how much ground area they occupy, and the proportion of the study area covered by detected buildings. These measures can act as simple indicators of surrounding development intensity. The polygons do **not** identify building type, use, occupancy, entrances, parking spaces, pedestrian access or population, so this notebook does not claim or estimate those attributes.

## Features produced by this notebook

For each `site_id`, the notebook measures detected buildings within an exact **500 m straight-line buffer** around the site coordinate and writes the following independent feature table:

| Output field | Plain meaning |
|---|---|
| `ms_building_count_500m` | Number of detected building footprints that overlap the 500 m buffer by a positive area. It describes the amount of separate detected building geometry around the site. |
| `ms_building_count_per_sqkm_500m` | The footprint count divided by the buffer area in square kilometres. It expresses detected building frequency on a common area basis. |
| `ms_building_footprint_area_m2_500m` | Total ground area, in square metres, covered by the union of detected building footprints inside the buffer. Parts outside the buffer are excluded and overlapping polygons are not double-counted. |
| `ms_building_coverage_ratio_500m` | Building-footprint area divided by total buffer area. For example, `0.10` means detected building polygons cover 10% of the 500 m buffer. |
| `ms_building_data_status` | Processing-status field showing whether calculation succeeded with buildings, succeeded with no detected buildings, or could not be completed because a required source partition was unavailable. It is a data-quality field, not an environmental measurement. |

All sites currently use the same 500 m buffer. Consequently, count and count per square kilometre contain the same relative information, while footprint area and coverage ratio contain the same relative information. Each pair should not be entered together as independent predictors in the same model.

## Scope and interpretation

This notebook reads the 49 authoritative sites from `data/processed/model_dataset.xlsx`, uses `site_id` as the sole join key, and outputs only the Microsoft status and features. Targets, QA fields and unrelated location attributes remain in `model_dataset.xlsx`. A detected footprint is not a confirmed cadastral building: completeness, positional accuracy and imagery date can vary by location. A zero count means that this release contains no detected footprint in the buffer; it does not prove that no building exists in reality.

## Reusable implementation and regression role

The reviewed scientific definitions remain in this notebook, but reusable source acquisition, validation, cleaning, spatial calculation, feature construction, source-specific QA and safe provenance now live in `src/pedflow/context/sources/microsoft_building_footprints.py`. The notebook supplies the validated `sites` table to that module and remains the human-readable scientific explanation and reviewed-output regression reference. The module does not read `model_dataset.xlsx` or write this workbook.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from openpyxl.styles import Alignment, Font, PatternFill


def find_project_root() -> Path:
    """Find the repository root from any supported working directory."""
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src" / "pedflow").is_dir():
            return path
    raise FileNotFoundError(
        f"Project root not found above {current}. Expected pyproject.toml and src/pedflow/."
    )


PROJECT_ROOT = find_project_root()
SOURCE_PACKAGE = PROJECT_ROOT / "src"
if str(SOURCE_PACKAGE) not in sys.path:
    sys.path.insert(0, str(SOURCE_PACKAGE))

from pedflow.context.contracts import SourceConfig
from pedflow.context.sources.microsoft_building_footprints import (
    FEATURE_COLUMNS,
    extract,
)
from pedflow.context.validation import (
    validate_and_normalise_sites,
    validate_source_features,
)


MODEL_DATASET_FILE = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "02_feature_extraction"
FEATURE_OUTPUT_FILE = (
    OUTPUT_DIR / "02_07_haoqi_microsoft_building_footprints_features.xlsx"
)
SOURCE_CONFIG = SourceConfig(
    name="microsoft_building_footprints",
    enabled=False,
    options={
        "source_release": "2026-07-24",
        "country": "Australia",
        "buffer_radius_m": 500,
        "quadkey_zoom": 9,
        "cache_directory": (
            "data/external_raw/microsoft_building_footprints/2026-07-24"
        ),
        "connect_timeout_seconds": 30,
        "read_timeout_seconds": 300,
        "max_retries": 3,
        "allow_download": True,
    },
)

if not MODEL_DATASET_FILE.is_file():
    raise FileNotFoundError(
        f"Model dataset not found: {MODEL_DATASET_FILE}\n"
        "Expected the authoritative workbook at data/processed/model_dataset.xlsx"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root  :", PROJECT_ROOT)
print("Model dataset :", MODEL_DATASET_FILE.relative_to(PROJECT_ROOT))
print("Feature output:", FEATURE_OUTPUT_FILE.relative_to(PROJECT_ROOT))


Project root  : /workspace
Model dataset : data/processed/model_dataset.xlsx
Feature output: 06_outputs/02_feature_extraction/02_07_haoqi_microsoft_building_footprints_features.xlsx


## 1. Read `Model_Data` and select source partitions

The authoritative `Model_Data` sheet contains 49 surveyed sites. This notebook reads only `site_id`, `latitude`, `longitude` and `coordinate_crs`; the pedestrian targets, `qa_status` and other project columns are not copied into the feature table. `site_id` is validated as a non-empty, unique string so every later team merge can use `validate="one_to_one"`.

The analysis uses a **500 m straight-line point buffer** as a transparent pilot parameter. It is not a confirmed client requirement, a walking-network catchment, or an along-road corridor. A local UTM CRS is estimated separately for each location before buffering so metre and area calculations are not performed in longitude/latitude degrees. Future analysis can compare radii or replace the point with a validated road-segment geometry without changing the source-download principle.

The official index fields used here are: `Location`, the source region; `QuadKey`, the level-9 Bing partition key; `Url`, the partition download address; `Size`, the publisher's compressed-size label; and `UploadDate`, the source upload date. `QuadKey` is an identifier and must remain text. Bounding boxes select candidate partitions conservatively; exact buffer intersections are applied later.

In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_FILE,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)

# Input validation belongs to the shared pipeline. The source adapter receives
# this validated four-column table and never reopens the modelling workbook.
sites = validate_and_normalise_sites(model_data)
result = extract(sites, SOURCE_CONFIG, PROJECT_ROOT)
building_features = validate_source_features(
    source_name=result.source_name,
    features=result.features,
    expected_site_ids=sites["site_id"],
    column_prefixes=("ms_",),
)

display(sites)
print(f"Sites supplied to the promoted module: {len(sites)}")
print(f"Required Australia tiles: {result.metadata['selected_tile_count']}")
print(f"Required quadkeys not listed: {result.metadata['missing_quadkey_count']}")


,site_id,latitude,longitude,coordinate_crs
0,250303-01,-32.866282,115.705163,EPSG:4326
1,250303-02,-33.359642,115.678041,EPSG:4326
2,250303-03,-33.360624,116.154738,EPSG:4326
3,250303-04,-34.238746,116.149491,EPSG:4326
4,250303-05,-32.628918,115.874612,EPSG:4326
5,250303-06,-33.082506,115.905724,EPSG:4326
6,250303-07,-33.574458,115.824750,EPSG:4326
7,250303-08,-33.553354,115.563265,EPSG:4326
8,250303-09,-32.845772,115.922622,EPSG:4326
9,250303-10,-33.959023,116.136902,EPSG:4326


Sites supplied to the promoted module: 49
Required Australia tiles: 33
Required quadkeys not listed: 0


## 2. Download and parse only the selected building files

Each selected URL is an immutable source partition for the pinned release. The compressed filename ends in `.csv.gz`, but its contents are GeoJSON Features separated by newline, not a conventional comma-separated table. Each feature contains a polygon geometry and a `properties` object.

Only `geometry` is used in this pilot. The sampled Australia records also contain `height` and `confidence`, but observed values include `height = 0` and `confidence = -1`. Without a sufficiently clear field definition and a local missingness/validity assessment, treating those values as measured height or model probability would be unsafe. They are therefore deliberately excluded rather than converted into weak features.

The source files are preserved unchanged in the ignored raw-data directory. Parsing extracts geometry in memory; it does not rewrite the downloaded partitions. If a required quadkey is absent from the release index, the affected site is reported as unavailable instead of being assigned a false zero.

In [3]:
source_run_summary = pd.Series(
    {
        "source_version": result.source_version,
        "retrieval_method": result.metadata["retrieval_method"],
        "cache_directory": result.metadata["cache_directory"],
        "selected_tile_count": result.metadata["selected_tile_count"],
        "cache_hit_count": result.metadata["cache_hit_count"],
        "download_count": result.metadata["download_count"],
        "source_geometry_count": result.metadata["source_geometry_count"],
        "index_sha256": result.metadata["index_sha256"],
        "tile_set_sha256": result.metadata["tile_set_sha256"],
    },
    name="value",
).to_frame()

display(source_run_summary)
if result.warnings:
    print("Source warnings:")
    for warning in result.warnings:
        print("-", warning)
else:
    print("Source warnings: none")


,value
source_version,2026-07-24
retrieval_method,Microsoft release CSV index and HTTPS partitio...
cache_directory,data/external_raw/microsoft_building_footprint...
selected_tile_count,33
cache_hit_count,34
download_count,0
source_geometry_count,824588
index_sha256,da7f619232c16f7b4e73fa7fd21d4ef805327f532ae037...
tile_set_sha256,b7294b62e107ff208d73be227f98f52aafe1cfbdde851d...


Source warnings:
- Repaired 4 invalid source geometries.


## 3. Calculate a small, interpretable feature set

A source footprint is counted when its intersection with the exact 500 m buffer has positive area. Footprint area is calculated after clipping to the buffer; the union of clipped polygons is used so overlapping detections cannot double-count covered ground. All spatial measurements use the site's local UTM CRS.

| Output field | Definition and interpretation |
|---|---|
| `ms_building_count_500m` | Number of source footprint records with positive-area overlap with the buffer. It can be affected by merged or split detections. |
| `ms_building_count_per_sqkm_500m` | Count divided by the exact buffer area in square kilometres. This is the normalised building-frequency measure. |
| `ms_building_footprint_area_m2_500m` | Union area of footprint geometry inside the buffer, in square metres. Boundary buildings contribute only their in-buffer area. |
| `ms_building_coverage_ratio_500m` | Footprint union area divided by buffer area. This is the normalised built-cover measure; for example, `0.10` means 10% of the buffer is covered by detected footprints. |
| `ms_building_data_status` | Distinguishes a completed calculation with buildings, a completed calculation with no detected footprints, and a required partition missing from the release index. |

Raw counts and areas are retained for auditability, while the per-square-kilometre count and coverage ratio are the more comparable forms. Because every row currently uses the same 500 m radius, count and count density are exact rescalings of one another; footprint area and coverage ratio are also exact rescalings. They should not be entered together as separate predictors in the same model. The normalised fields become essential if later buffers differ in area.

In [4]:
features = building_features.loc[:, list(FEATURE_COLUMNS)].copy()

# These are the promoted module outputs. The definitions and interpretation
# remain documented in the Markdown immediately above this cell.
display(features)
display(
    features["ms_building_data_status"]
    .value_counts(dropna=False)
    .to_frame("sites")
)


,site_id,ms_building_data_status,ms_building_count_500m,ms_building_count_per_sqkm_500m,ms_building_footprint_area_m2_500m,ms_building_coverage_ratio_500m
0,250303-01,complete_with_buildings,12,15.285,2320.979,0.002956
1,250303-02,complete_with_buildings,175,222.906,122504.370,0.156040
2,250303-03,complete_with_buildings,370,471.288,136177.905,0.173457
3,250303-04,complete_with_buildings,392,499.310,98962.098,0.126053
4,250303-05,complete_with_buildings,329,419.064,103975.996,0.132440
5,250303-06,complete_with_buildings,222,282.773,54410.108,0.069305
6,250303-07,complete_with_buildings,286,364.293,88338.909,0.112522
7,250303-08,complete_with_buildings,340,433.075,73080.979,0.093087
8,250303-09,complete_with_buildings,287,365.567,66195.305,0.084316
9,250303-10,complete_with_buildings,317,403.779,77878.571,0.099198


,sites
ms_building_data_status,
complete_with_buildings,49


## 4. Save one clear feature table

The output is an independent team feature table, not a copy of `model_dataset.xlsx`. It contains exactly one row per `site_id`, one source-status field and four building measures. It excludes coordinates, road fields, QA fields and pedestrian targets because those remain authoritative in `Model_Data`.

One workbook, `02_07_haoqi_microsoft_building_footprints_features.xlsx`, is written directly under `06_outputs/02_feature_extraction/`. Its name follows the notebook stem `02_07_haoqi_microsoft_building_footprints`, with `_features` added to identify the derived table. Its only worksheet is `Building_Features`, so `pd.read_excel(path)` returns the feature table directly. Later modelling code merges it on `site_id` with `how="left"` and `validate="one_to_one"`.

In [5]:
# The existing workbook is the reviewed regression reference. Compare the
# promoted module result before replacing it with an identically structured file.
if FEATURE_OUTPUT_FILE.is_file():
    reviewed_features = pd.read_excel(
        FEATURE_OUTPUT_FILE,
        sheet_name="Building_Features",
        dtype={"site_id": "string"},
    )
    pd.testing.assert_frame_equal(
        features[["site_id", "ms_building_data_status"]].reset_index(drop=True),
        reviewed_features[["site_id", "ms_building_data_status"]].reset_index(drop=True),
        check_dtype=False,
    )
    pd.testing.assert_series_equal(
        features["ms_building_count_500m"].reset_index(drop=True),
        reviewed_features["ms_building_count_500m"].reset_index(drop=True),
        check_dtype=False,
        check_exact=True,
    )
    for column, tolerance in {
        "ms_building_count_per_sqkm_500m": 0.001,
        "ms_building_footprint_area_m2_500m": 0.001,
        "ms_building_coverage_ratio_500m": 0.000001,
    }.items():
        pd.testing.assert_series_equal(
            features[column].reset_index(drop=True),
            reviewed_features[column].reset_index(drop=True),
            check_dtype=False,
            check_exact=False,
            rtol=0,
            atol=tolerance,
        )
    print("Regression check: promoted features match the reviewed workbook.")

with pd.ExcelWriter(FEATURE_OUTPUT_FILE, engine="openpyxl") as writer:
    features.to_excel(writer, sheet_name="Building_Features", index=False)
    worksheet = writer.sheets["Building_Features"]
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    worksheet.sheet_view.showGridLines = False

    header_fill = PatternFill("solid", fgColor="1F4E78")
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = Font(color="FFFFFF", bold=True)
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )
    worksheet.row_dimensions[1].height = 42

    for column, width in {
        "A": 14,
        "B": 31,
        "C": 27,
        "D": 36,
        "E": 39,
        "F": 36,
    }.items():
        worksheet.column_dimensions[column].width = width
    for row in range(2, len(features) + 2):
        worksheet[f"C{row}"].number_format = "0"
        worksheet[f"D{row}"].number_format = "0.000"
        worksheet[f"E{row}"].number_format = "0.000"
        worksheet[f"F{row}"].number_format = "0.000000"

saved_features = pd.read_excel(
    FEATURE_OUTPUT_FILE,
    sheet_name="Building_Features",
    dtype={"site_id": "string"},
)
pd.testing.assert_frame_equal(
    saved_features,
    features.astype({"site_id": "string"}),
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9,
)

display(features)
print("Saved:", FEATURE_OUTPUT_FILE.relative_to(PROJECT_ROOT))


Regression check: promoted features match the reviewed workbook.


,site_id,ms_building_data_status,ms_building_count_500m,ms_building_count_per_sqkm_500m,ms_building_footprint_area_m2_500m,ms_building_coverage_ratio_500m
0,250303-01,complete_with_buildings,12,15.285,2320.979,0.002956
1,250303-02,complete_with_buildings,175,222.906,122504.370,0.156040
2,250303-03,complete_with_buildings,370,471.288,136177.905,0.173457
3,250303-04,complete_with_buildings,392,499.310,98962.098,0.126053
4,250303-05,complete_with_buildings,329,419.064,103975.996,0.132440
5,250303-06,complete_with_buildings,222,282.773,54410.108,0.069305
6,250303-07,complete_with_buildings,286,364.293,88338.909,0.112522
7,250303-08,complete_with_buildings,340,433.075,73080.979,0.093087
8,250303-09,complete_with_buildings,287,365.567,66195.305,0.084316
9,250303-10,complete_with_buildings,317,403.779,77878.571,0.099198


Saved: 06_outputs/02_feature_extraction/02_07_haoqi_microsoft_building_footprints_features.xlsx
